### Setup

In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(cowplot)
  library(ggpubr)
  library(grid)
  library(ggplot2)
})

AMP_COL    <- "#6D2682"
GROWTH_COL <- "#2BBFD9"

# Nonzero ampicillin concentrations used during the pre-pulse.
# These are exposure concentrations, not MIC thresholds.
PRE_PULSE_CONC_UG_ML <- c(5, 50, 500)

out_dir <- "../figs"

FS_ALL    <- 26
TEXT_GEOM <- 8.5

pSpacer <- ggplot() +
  theme_void() +
  theme(
    plot.background  = element_rect(fill = "white", colour = NA),
    panel.background = element_rect(fill = "white", colour = NA)
  )

# species labels without HAMBI codes
scale_labels <- c(
  expression(italic('Chitinophaga sancti')),
  expression(italic('Paracoccus denitrificans')),
  expression(italic('Moraxella canis')),
  expression(italic('Microvirga lotononidis')),
  expression(italic('Bordetella avium')),
  expression(italic('Niabella yanshanensis')),
  expression(italic('Hafnia alvei')),
  expression(italic('Morganella morganii')),
  expression(italic('Acinetobacter johnsonii')),
  expression(italic('Sphingobium yanoikuyae')),
  expression(italic('Brevundimonas bullata')),
  expression(italic('Kluyvera intermedia')),
  expression(italic('Cupriavidus oxalaticus')),
  expression(italic('Paraburkholderia kururiensis')),
  expression(italic('Agrobacterium tumefaciens')),
  expression(italic('Stenotrophomonas maltophilia')),
  expression(italic('Citrobacter koseri')),
  expression(italic('Pseudomonas putida')),
  expression(italic('Sphingobacterium spiritivorum')),
  expression(italic('Aeromonas caviae')),
  expression(italic('Comamonas testosteroni')),
  expression(italic('Pseudomonas chlororaphis')),
  expression(italic('Trinickia caryophylli'))
)

### Panel A data (MIC + carrying capacity)

In [2]:
df <- read.table("../data/AMP_growth_k_auc.txt", header = TRUE, check.names = FALSE, sep = "\t")
df$k <- ifelse(df$k < 0.1 | df$k > 1.3, 0, df$k)
df <- df[df$Species != "only medium", ]

df <- df %>%
  mutate(strainID = paste0("HAMBI-", str_pad(Species, 4, pad = "0"))) %>%
  select(strainID, AB, Pop, Replicate, k)

df$Treat <- with(df, paste(strainID, Pop, Replicate, sep = "_"))
Treatments <- unique(df$Treat)

# MIC per clone:
# MIC is the next measured AB concentration above the highest concentration
# where growth was observed.
# Positive growth values after two successive zero-growth values are ignored as likely errors.

df$MIC <- NA_real_

for (i in seq_along(Treatments)) {
  
  a <- df[df$Treat == Treatments[i], ]
  a <- a[order(a$AB), ]
  
  # Store original measured concentration scale before filtering
  full_AB <- sort(unique(a$AB))
  
  # Find first concentration where two consecutive no-growth values occur
  first_confirmed_zero <- NA_real_
  
  if (nrow(a) >= 2) {
    for (j in seq_len(nrow(a) - 1)) {
      if (a$k[j] == 0 && a$k[j + 1] == 0) {
        first_confirmed_zero <- a$AB[j]
        break
      }
    }
  }
  
  # Use only values before confirmed no-growth region to identify real growth
  if (!is.na(first_confirmed_zero)) {
    a_growth_region <- a[a$AB < first_confirmed_zero, ]
  } else {
    a_growth_region <- a
  }
  
  # Highest concentration with growth before confirmed no-growth region
  growth_AB <- a_growth_region$AB[a_growth_region$k > 0]
  
  if (length(growth_AB) == 0) {
    
    if (!is.na(first_confirmed_zero)) {
      a_MIC <- first_confirmed_zero
    } else {
      a_MIC <- min(full_AB, na.rm = TRUE)
    }
    
  } else {
    
    last_growth <- max(growth_AB, na.rm = TRUE)
    
    # MIC is next measured concentration in the original AB scale
    next_AB <- full_AB[full_AB > last_growth]
    
    if (length(next_AB) > 0) {
      a_MIC <- min(next_AB, na.rm = TRUE)
    } else {
      # Growth even at highest tested concentration
      a_MIC <- max(full_AB, na.rm = TRUE)
    }
  }
  
  df$MIC[df$Treat == Treatments[i]] <- a_MIC
}

# keep one row per clone + AB==0 k
df2  <- df[!duplicated(df$Treat), c("strainID", "Pop", "Replicate", "Treat", "MIC")]
df_k <- df[df$AB == 0, c("strainID", "Pop", "Replicate", "Treat", "k")]
df3  <- merge(df2, df_k, by = c("strainID", "Pop", "Replicate", "Treat"))

df3$log10MIC <- log10(df3$MIC + 1e-5)

# species order by ancestral MIC
df4 <- aggregate(cbind(log10MIC, k) ~ strainID + Pop, data = df3, FUN = mean, na.rm = TRUE)
speciesord <- df4 %>% dplyr::filter(Pop == "ANC") %>% arrange(log10MIC) %>% pull(strainID)
names(scale_labels) <- speciesord

df3$strainID <- factor(df3$strainID, levels = speciesord)
df3$Pop <- factor(
  df3$Pop,
  levels = c("ANC", "EVO"),
  labels = c("Ancestral", "Resistance primed")
)

df7 <- bind_rows(
  df3 %>% transmute(strainID, Pop, Replicate, Trait = log10MIC, Trait_type = "log10MIC"),
  df3 %>% transmute(strainID, Pop, Replicate, Trait = k,       Trait_type = "k")
)

df7$Trait_type <- factor(
  df7$Trait_type,
  levels = c("log10MIC", "k")
)

# Remove species without valid positive MIC or carrying-capacity values
excluded_species <- c(
  "HAMBI-1988",  # Chitinophaga sancti
  "HAMBI-3237"   # Microvirga lotononidis
)

df7A <- df7 %>%
  filter(!strainID %in% excluded_species)

speciesord_A <- speciesord[
  !speciesord %in% excluded_species
]

df7A$strainID <- factor(
  df7A$strainID,
  levels = speciesord_A
)

# Mean and standard error across technical replicates
df7A_sum <- df7A %>%
  group_by(strainID, Pop, Trait_type) %>%
  summarise(
    n = sum(is.finite(Trait)),
    mean = mean(Trait, na.rm = TRUE),
    se = if_else(
      n > 1,
      sd(Trait, na.rm = TRUE) / sqrt(n),
      NA_real_
    ),
    .groups = "drop"
  )

pop_fill <- c(
  "Ancestral" = "white",
  "Resistance primed" = "grey15"
)

pop_shape <- c(
  "Ancestral" = 21,
  "Resistance primed" = 24
)

# Reference lines and labels for the three nonzero pre-pulse
# ampicillin concentrations.
pre_pulse_lines <- tibble(
  yint = log10(PRE_PULSE_CONC_UG_ML),
  label_y = log10(PRE_PULSE_CONC_UG_ML) + 0.18,
  label = c(
    "5~mu*g~mL^{-1}",
    "50~mu*g~mL^{-1}",
    "500~mu*g~mL^{-1}"
  )
)

make_trait_panel <- function(
    trait_name,
    trait_colour,
    y_label,
    show_species_labels = TRUE,
    show_pre_pulse_lines = FALSE
) {

  plot_data <- df7A %>%
    filter(Trait_type == trait_name)

  plot_summary <- df7A_sum %>%
    filter(Trait_type == trait_name)

  p <- ggplot(
    plot_data,
    aes(x = strainID, y = Trait)
  )

  # Draw dashed reference lines behind the observations
  if (show_pre_pulse_lines) {
    p <- p +
      geom_hline(
        data = pre_pulse_lines,
        aes(yintercept = yint),
        inherit.aes = FALSE,
        colour = AMP_COL,
        linetype = 2,
        linewidth = 0.9
      )
  }

  p <- p +

    # Technical-replicate observations
    geom_point(
      aes(shape = Pop, fill = Pop),
      colour = trait_colour,
      alpha = 0.22,
      size = 3.6,
      stroke = 1.4,
      show.legend = FALSE
    ) +

    # Mean ± standard error
    geom_errorbar(
      data = plot_summary,
      aes(
        x = strainID,
        y = mean,
        ymin = mean - se,
        ymax = mean + se
      ),
      inherit.aes = FALSE,
      colour = trait_colour,
      width = 0.12,
      linewidth = 1.3,
      show.legend = FALSE
    ) +

    geom_point(
      data = plot_summary,
      aes(
        x = strainID,
        y = mean,
        shape = Pop,
        fill = Pop
      ),
      inherit.aes = FALSE,
      colour = trait_colour,
      size = 5.2,
      stroke = 1.8,
      show.legend = FALSE
    ) +

    scale_fill_manual(
      values = pop_fill,
      guide = "none"
    ) +

    scale_shape_manual(
      values = pop_shape,
      guide = "none"
    ) +

    scale_x_discrete(
  labels = scale_labels[speciesord_A],
  expand = expansion(add = 0.30)
) +

    coord_cartesian(clip = "off") +

    labs(
      x = NULL,
      y = y_label
    ) +

    theme_bw(base_size = FS_ALL) +

    theme(
      panel.grid = element_blank(),

      panel.background = element_rect(
        fill = "white",
        colour = NA
      ),

      plot.background = element_rect(
        fill = "white",
        colour = NA
      ),

      panel.border = element_rect(
        colour = "black",
        fill = NA,
        linewidth = 1.2
      ),

      axis.text.x = element_text(
        angle = 40,
        hjust = 1,
        vjust = 1,
        size = FS_ALL,
        colour = "black"
      ),

      axis.text.y = element_text(
        size = FS_ALL,
        colour = "black"
      ),

      axis.title.y = element_text(
      size = FS_ALL,
      face = "plain",
      colour = "black",
      margin = margin(r = 14)
    ),

      legend.position = "none",

      # A genuine left margin for the y-axis title and tick labels
      plot.margin = margin(
        t = 8,
        r = 14,
        b = 12,
        l = 48
      )
    )

  if (show_pre_pulse_lines) {
    p <- p +
      geom_text(
  data = pre_pulse_lines,
  aes(
    x = 1,
    y = label_y,
    label = label
  ),
  inherit.aes = FALSE,
  hjust = 0,
  vjust = 0,
  size = TEXT_GEOM,
  colour = "black",
  parse = TRUE
) +

      # Extra lower expansion prevents the label under the lowest
      # dashed line from being crowded.
      scale_y_continuous(
        expand = expansion(mult = c(0.10, 0.07))
      )
  }

  if (trait_name == "k") {
    p <- p +
      scale_y_continuous(
        # Exactly one decimal on carrying-capacity tick labels
        labels = function(x) sprintf("%.1f", x),
        expand = expansion(mult = c(0.07, 0.09))
      )
  }

  if (!show_species_labels) {
    p <- p +
      theme(
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),

        # Leave a gap between A and B for the B panel tag
        plot.margin = margin(
          t = 8,
          r = 14,
          b = 18,
          l = 48
        )
      )
  } else {
    p <- p +
      theme(
        plot.margin = margin(
          t = 18,
          r = 14,
          b = 12,
          l = 48
        )
      )
  }

  p
}

# Panel A: MIC
pA <- make_trait_panel(
  trait_name = "log10MIC",
  trait_colour = AMP_COL,
  y_label = expression(
    "MIC (" * log[10] ~ mu * plain(g) ~ plain(mL)^{-1} * ")"
  ),
  show_species_labels = FALSE,
  show_pre_pulse_lines = TRUE
)

# Panel B: carrying capacity
pB <- make_trait_panel(
  trait_name = "k",
  trait_colour = GROWTH_COL,
  y_label = expression(
    "Carrying capacity " * italic(k) ~
      "(" * plain(OD)[600] * ")"
  ),
  show_species_labels = TRUE,
  show_pre_pulse_lines = FALSE
)

### Panel C: mutation heatmap (remove hypothetical/NA variants) + panel border

In [3]:
wgs <- read.table("../data/curated_wgs_nonsyn_mutations.txt", header = TRUE, sep = "\t")
sample_order <- c("HAMBI_1972","HAMBI_2160","HAMBI_2659","HAMBI_0105","HAMBI_1977")

variant_order <- rev(c(
  "H1972_00439_ftsI_p.Gly537Val",
  "H1972_01245_erfK_p.Thr229Pro",
  "H1972_03930_rpoA_p.Ala308Pro",
  "H2160_00934_cysS_p.Arg307His",
  "H2659_00656_smf-1_p.Ala9fs",
  "H0105_04901_rbsA_6_p.Val387Glu",
  "H1977_01749_rne_p.Arg518fs",
  "H1977_03891_ampR_p.Asp135Asn"
))

scale_labels2 <- c(
  expression(italic('Aeromonas caviae')),
  expression(italic('Bordetella avium')),
  expression(italic('Stenotrophomonas maltophilia')),
  expression(italic('Agrobacterium tumefaciens')),
  expression(italic('Pseudomonas chlororaphis'))
)

extract_gene <- function(v) {
  v %>% str_replace(".*_[0-9]{5}_", "") %>% str_replace("_p\\..*$", "")
}

heat_long <- wgs %>%
  filter(Variant %in% variant_order) %>%
  transmute(
    Sample  = factor(Sample, levels = sample_order),
    Variant = factor(Variant, levels = variant_order),
    Hit = 1L
  ) %>%
  tidyr::complete(Sample, Variant, fill = list(Hit = 0L)) %>%
  mutate(
    gene = extract_gene(as.character(Variant)),
    Hit  = factor(Hit, levels = c(0,1))
  ) %>%
  filter(!is.na(gene), gene != "", gene != "NA")

pC <- ggplot(
  heat_long,
  aes(x = Sample, y = Variant, fill = Hit)
) +
  geom_tile(
    colour = "grey92",
    linewidth = 0.65
  ) +

  scale_fill_manual(
    values = c(
      "0" = "white",
      "1" = AMP_COL
    ),
    guide = "none"
  ) +

  scale_x_discrete(
    labels = scale_labels2,
    expand = expansion(add = 0.02)
  ) +

  scale_y_discrete(
    position = "right",
    labels = function(v) {
      gene_names <- extract_gene(v)

      as.expression(
        lapply(
          gene_names,
          function(gene) {
            bquote(italic(.(gene)))
          }
        )
      )
    }
  ) +

  labs(
    y = "Gene with nonsynonymous mutation",
    x = NULL
  ) +

  theme_bw(base_size = FS_ALL) +

  theme(
    panel.grid = element_blank(),

    panel.background = element_rect(
      fill = "white",
      colour = NA
    ),

    plot.background = element_rect(
      fill = "white",
      colour = NA
    ),

    panel.border = element_rect(
      colour = "black",
      fill = NA,
      linewidth = 1.2
    ),

    axis.text.x = element_text(
      angle = 40,
      hjust = 1,
      vjust = 1,
      size = FS_ALL,
      colour = "black"
    ),

    axis.title.y = element_text(
      face = "plain",
      size = FS_ALL,
      colour = "black"
    ),

    axis.text.y = element_text(
      size = FS_ALL,
      colour = "black"
    ),

    legend.position = "none",

    plot.margin = margin(
      t = 8,
      r = 22,
      b = 12,
      l = 12
    )
  )

### Combine ABC

In [4]:
# Shape-only legend for ancestral and resistance-primed isolates
legend_data <- tibble(
  Pop = factor(
    c("Ancestral", "Resistance primed"),
    levels = c("Ancestral", "Resistance primed")
  ),
  x = c(1, 2),
  y = c(1, 1)
)

legend_source <- ggplot(
  legend_data,
  aes(x = x, y = y, shape = Pop, fill = Pop)
) +
  geom_point(
    size = 5.2,
    stroke = 1.8,
    colour = "black"
  ) +
  scale_shape_manual(
    values = pop_shape,
    name = NULL
  ) +
  scale_fill_manual(
    values = pop_fill,
    guide = "none"
  ) +
  guides(
    shape = guide_legend(
      title = NULL,
      override.aes = list(size = 5.2)
    )
  ) +
  theme_void(base_size = FS_ALL) +
  theme(
    legend.position = "top",
    legend.text = element_text(size = FS_ALL),

    # Reduce space around the extracted symbol legend
    legend.margin = margin(
      t = 0,
      r = 0,
      b = -14,
      l = 0
    ),

    legend.box.margin = margin(
      t = 0,
      r = 0,
      b = 0,
      l = 0
    ),

    plot.margin = margin(
      t = 0,
      r = 0,
      b = 0,
      l = 0
    )
  )

global_legend <- get_legend(legend_source)

# Relative panel heights within the A/B stack
A_REL_HEIGHT <- 0.84
B_REL_HEIGHT <- 1.16

trait_stack_no_tags <- plot_grid(
  pA,
  pB,
  ncol = 1,
  rel_heights = c(A_REL_HEIGHT, B_REL_HEIGHT),
  align = "v",
  axis = "lr"
)

# Draw the plots inside only 92% of the available height.
# This creates real space above A.
STACK_DRAW_HEIGHT <- 0.92

# Top of B inside the combined A/B coordinate system
B_PANEL_TOP <- STACK_DRAW_HEIGHT *
  B_REL_HEIGHT /
  (A_REL_HEIGHT + B_REL_HEIGHT)

trait_panels <- ggdraw() +

  draw_plot(
    trait_stack_no_tags,
    x = 0,
    y = 0,
    width = 1,
    height = STACK_DRAW_HEIGHT
  ) +

  # A: substantially higher than the panel border
  draw_plot_label(
  label = "A",
  x = 0.0,
y = 0.95,
  size = FS_ALL + 8,
  fontface = "bold",
  hjust = 0,
  vjust = 1
)+

  # B: slightly above the B panel border
draw_plot_label(
  label = "B",
x = 0.0,
y = B_PANEL_TOP + 0.01,
  size = FS_ALL + 8,
  fontface = "bold",
  hjust = 0,
  vjust = 1
)

# C receives the same upper headroom as A
pC_labeled <- ggdraw() +

  draw_plot(
    pC,
    x = 0,
    y = 0,
    width = 1,
    height = 0.92
  ) +

  # Panel letter inside the upper-left corner of panel C
  draw_plot_label(
    label = "C",
x = -0.03,
y = 0.95,
    size = FS_ALL + 8,
    fontface = "bold",
    hjust = 0,
    vjust = 1
  )

panels <- plot_grid(
  trait_panels,
  pC_labeled,
  ncol = 2,
  rel_widths = c(0.74, 0.26)
)

# Add a real outer gutter so the left-side text cannot be cropped.
panels_with_outer_margin <- ggdraw() +
  draw_plot(
    panels,
    x = 0.030,
    y = 0,
    width = 0.955,
    height = 1
  )

final_plot <- ggdraw() +
  draw_plot(
    panels_with_outer_margin,
    x = 0,
    y = 0,
    width = 1,
    height = 0.94
  ) +
  draw_grob(
    global_legend,
    x = 0.36,
    y = 0.91,
    width = 0.30,
    height = 0.06
  )

pdf(
  file.path(out_dir, "Fig2.pdf"),
  width = 25,
  height = 14,
  bg = "white"
)

print(final_plot)
dev.off()

pdf 
  2

## Initial strain MIC and carrying capacity: ANOVA

In [5]:
# Read in data

df = read.table("../data/AMP_growth_k_auc.txt", header = T, check.names = F, sep = "\t")

# Convert unreliable k values to 0 = no growth

df$k = ifelse(df$k < 0.1 | df$k > 1.3, 0, df$k)

# Remove control measurements

df = df[df$Species != "only medium", ]

# Convert strain numbers to HAMBI codes

df = df %>%
  mutate(strainID = paste0("HAMBI-", str_pad(Species, 4, pad = c("0"))))

df = df[, c("strainID", "AB", "Pop", "Replicate", "k")]

# Add treatment information

df$Treat = paste(df$strainID, df$Pop, df$Replicate, sep = "_")

# Create vector for treatments

Treatments = unique(df$Treat)

# For each unique clone, compute MIC as the next measured AB concentration
# above the highest concentration where growth was observed.
# Positive growth values after two successive zero-growth values are ignored as errors.

df$MIC = NA_real_

for(i in seq_along(Treatments)) {
  
  a = df[df$Treat == Treatments[i], ]
  a = a[order(a$AB), ]
  
  # Store the full measured concentration scale before any filtering
  full_AB = sort(unique(a$AB))
  
  # Find the first concentration where two consecutive zero-growth values occur
  first_confirmed_zero = NA_real_
  
  if(nrow(a) >= 2) {
    for(j in seq_len(nrow(a) - 1)) {
      if(a$k[j] == 0 & a$k[j + 1] == 0) {
        first_confirmed_zero = a$AB[j]
        break
      }
    }
  }
  
  # Use only values before the confirmed no-growth region to identify real growth
  if(!is.na(first_confirmed_zero)) {
    a_growth_region = a[a$AB < first_confirmed_zero, ]
  } else {
    a_growth_region = a
  }
  
  # Highest concentration with growth before the confirmed no-growth region
  growth_AB = a_growth_region$AB[a_growth_region$k > 0]
  
  if(length(growth_AB) == 0) {
    
    # No growth before the confirmed no-growth region.
    # MIC is the first confirmed no-growth concentration if available;
    # otherwise it is the lowest measured concentration.
    if(!is.na(first_confirmed_zero)) {
      a_MIC = first_confirmed_zero
    } else {
      a_MIC = min(full_AB, na.rm = T)
    }
    
  } else {
    
    last_growth = max(growth_AB, na.rm = T)
    
    # MIC is the next measured concentration in the original AB scale
    next_AB = full_AB[full_AB > last_growth]
    
    if(length(next_AB) > 0) {
      a_MIC = min(next_AB, na.rm = T)
    } else {
      # Growth even at the highest tested concentration
      a_MIC = max(full_AB, na.rm = T)
    }
  }
  
  df$MIC[df$Treat == Treatments[i]] = a_MIC
}

# Remove duplicate rows

df2 = df[!duplicated(df$Treat), c("strainID", "Pop", "Replicate", "Treat", "MIC")]

# Add k for antibiotic-free condition

df_k = df[df$AB == 0, c("strainID", "Pop", "Replicate", "Treat", "k")]

df3 = merge(df2, df_k, by = c("strainID", "Pop", "Replicate", "Treat"))

# Reorder

df3 = df3[order(df3$strainID, df3$Pop), ]

# Add very small random noise to allow log conversion and statistical testing of 0 values

set.seed(1)
df3$noise = runif(nrow(df3), 1e-5, 1e-4)

df3$log10MIC = log10(df3$MIC + df3$noise)
df3$MIC_noise = df3$MIC + df3$noise
df3$k_noise = df3$k + df3$noise

M_MIC = lm(log10MIC ~ strainID * Pop, data = df3)

aov_mic = anova(M_MIC)
rownames(aov_mic) = c(
  "Strain",
  "Ampicillin pre-exposure",
  "Strain x ampicillin pre-exposure",
  "Residuals"
)
aov_mic = cbind("Factor" = rownames(aov_mic), aov_mic)
aov_mic

M_k = lm(k ~ strainID * Pop, data = df3)

aov_k = anova(M_k)
rownames(aov_k) = c(
  "Strain",
  "Ampicillin pre-exposure",
  "Strain x ampicillin pre-exposure",
  "Residuals"
)
aov_k = cbind("Factor" = rownames(aov_k), aov_k)
aov_k

# Check species with highest effect size

df3_effect_anc = aggregate(
  data = df3[df3$Pop == "ANC", c("strainID", "Replicate", "log10MIC", "k")],
  cbind(log10MIC, k) ~ strainID,
  FUN = mean
)
colnames(df3_effect_anc) = c("strainID", "log10MIC_ANC", "k_ANC")

df3_effect_evo = aggregate(
  data = df3[df3$Pop == "EVO", c("strainID", "Replicate", "log10MIC", "k")],
  cbind(log10MIC, k) ~ strainID,
  FUN = mean
)
colnames(df3_effect_evo) = c("strainID", "log10MIC_EVO", "k_EVO")

df3_effect = merge(df3_effect_anc, df3_effect_evo, by = "strainID")

df3_effect$log10MIC_ratio = df3_effect$log10MIC_EVO / df3_effect$log10MIC_ANC
df3_effect$k_ratio = df3_effect$k_EVO / df3_effect$k_ANC

head(df3_effect)

,Factor,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Strain,Strain,22,250.598398,11.3908363,48.638855,8.672141e-25
Ampicillin pre-exposure,Ampicillin pre-exposure,1,6.653933,6.6539328,28.412284,2.879072e-06
Strain x ampicillin pre-exposure,Strain x ampicillin pre-exposure,22,34.660067,1.5754576,6.727202,2.979092e-08
Residuals,Residuals,46,10.772837,0.2341921,NA,NA


,Factor,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Strain,Strain,22,3.88141910,0.176428141,18.054842,7.644499e-16
Ampicillin pre-exposure,Ampicillin pre-exposure,1,0.04856283,0.048562826,4.969695,3.071970e-02
Strain x ampicillin pre-exposure,Strain x ampicillin pre-exposure,22,1.13724630,0.051693014,5.290024,9.932510e-07
Residuals,Residuals,46,0.44950239,0.009771791,NA,NA


,strainID,log10MIC_ANC,k_ANC,log10MIC_EVO,k_EVO,log10MIC_ratio,k_ratio
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,HAMBI-0006,1.62816345,0.8215607,3.3010300,0.5753560,2.0274562,0.7003208
2,HAMBI-0097,0.04336306,1.1176546,-0.2207358,0.5416475,-5.0904114,0.4846288
3,HAMBI-0105,0.83575497,0.5974698,0.9237994,0.5340331,1.1053472,0.8938243
4,HAMBI-0262,0.30749104,0.2981430,0.9238014,0.2794971,3.0043197,0.9374599
5,HAMBI-0403,2.77275628,0.1961860,2.6847106,0.2362334,0.9682462,1.2041295
6,HAMBI-1279,-0.22075070,0.1636622,2.0683913,0.2181412,-9.3698064,1.3328745


### Ampicillin degradation analysis and stats

In [6]:
# ---- Load and filter ----
dat <- read.table("../data/AMP_key_species.txt", header=TRUE, sep="\t", dec=",") %>%
  filter(Species == "HAMBI_1972") %>%
  mutate(
    Evo   = factor(Evo, levels=c("ANC","EVO"),
                   labels=c("Ancestral","Pre-exposed")),
    Timeh = as.numeric(Timeh),
    AMP_conc = as.numeric(AMP_conc),
    OD = as.numeric(OD)
  )

# ---- Fixed baseline per Evo ----
init_by_evo <- dat %>%
  filter(Timeh == 0) %>%
  group_by(Evo) %>%
  summarise(init = mean(AMP_conc, na.rm=TRUE), .groups="drop")

dat <- dat %>%
  left_join(init_by_evo, by="Evo") %>%
  mutate(AMP_rel = AMP_conc / init)

# ---- Absolute threshold < 10 µg/mL ----
deg_time_abs <- dat %>%
  group_by(Evo, Replicate) %>%
  summarise(Time_to_deg = min(Timeh[AMP_conc < 10], na.rm = TRUE), .groups = "drop") %>%
  mutate(Time_to_deg = ifelse(is.infinite(Time_to_deg), NA, Time_to_deg))

print(deg_time_abs)

# ---- Correct Wilcoxon (Pre-exposed earlier) ----
wilx <- wilcox.test(
  x = deg_time_abs$Time_to_deg[deg_time_abs$Evo == "Pre-exposed"],
  y = deg_time_abs$Time_to_deg[deg_time_abs$Evo == "Ancestral"],
  alternative = "less",   # now correct direction
  exact = TRUE, correct = FALSE
)
wilx

# ---- Fisher’s exact at 6 h (<10 µg/mL) ----
deg6_abs <- dat %>%
  filter(Timeh == 6) %>%
  mutate(Degraded10 = AMP_conc < 10)

tab6 <- table(deg6_abs$Evo, deg6_abs$Degraded10)
print(tab6)
print(fisher.test(tab6, alternative="greater"))

# ---- Welch t-test for OD ----
t_od <- t.test(OD ~ Evo, data = deg6_abs, var.equal = FALSE)
t_od

# ---- Correlation between degradation time and growth ----
summ_merge <- left_join(deg_time_abs,
                        deg6_abs %>% select(Evo, Replicate, OD6 = OD),
                        by=c("Evo","Replicate"))
cor.test(summ_merge$Time_to_deg, summ_merge$OD6, method="spearman")

# A tibble: 6 × 3
  Evo         Replicate Time_to_deg
  <fct>           <int>       <dbl>
1 Ancestral           1          12
2 Ancestral           2          12
3 Ancestral           3          12
4 Pre-exposed         1           6
5 Pre-exposed         2           6
6 Pre-exposed         3           6


Warning message in wilcox.test.default(x = deg_time_abs$Time_to_deg[deg_time_abs$Evo == :
“cannot compute exact p-value with ties”



	Wilcoxon rank sum test

data:  deg_time_abs$Time_to_deg[deg_time_abs$Evo == "Pre-exposed"] and deg_time_abs$Time_to_deg[deg_time_abs$Evo == "Ancestral"]
W = 0, p-value = 0.01267
alternative hypothesis: true location shift is less than 0


             
              FALSE TRUE
  Ancestral       3    0
  Pre-exposed     0    3

	Fisher's Exact Test for Count Data

data:  tab6
p-value = 0.05
alternative hypothesis: true odds ratio is greater than 1
95 percent confidence interval:
   1 Inf
sample estimates:
odds ratio 
       Inf 




	Welch Two Sample t-test

data:  OD by Evo
t = -29.309, df = 2.1514, p-value = 0.0007719
alternative hypothesis: true difference in means between group Ancestral and group Pre-exposed is not equal to 0
95 percent confidence interval:
 -0.2096488 -0.1590179
sample estimates:
  mean in group Ancestral mean in group Pre-exposed 
               0.04233333                0.22666667 


Warning message in cor.test.default(summ_merge$Time_to_deg, summ_merge$OD6, method = "spearman"):
“Cannot compute exact p-value with ties”



	Spearman's rank correlation rho

data:  summ_merge$Time_to_deg and summ_merge$OD6
S = 65.741, p-value = 0.02131
alternative hypothesis: true rho is not equal to 0
sample estimates:
       rho 
-0.8783101 


In [7]:
# =========================================================
# Degradation stats for the other two strains
# =========================================================

other_strains <- c("HAMBI_403", "HAMBI_1977")

for (sp in other_strains) {
  
  cat("\n============================================\n")
  cat("Strain:", sp, "\n")
  cat("============================================\n")
  
  dat_sp <- read.table("../data/AMP_key_species.txt", header=TRUE, sep="\t", dec=",") %>%
    filter(Species == sp) %>%
    mutate(
      Evo   = factor(Evo, levels = c("ANC", "EVO"),
                     labels = c("Ancestral", "Pre-exposed")),
      Timeh = as.numeric(Timeh),
      AMP_conc = as.numeric(AMP_conc),
      OD = as.numeric(OD)
    )
  
  # time to reach <10 ug/mL
  deg_time_abs_sp <- dat_sp %>%
    group_by(Evo, Replicate) %>%
    summarise(
      Time_to_deg = min(Timeh[AMP_conc < 10], na.rm = TRUE),
      .groups = "drop"
    ) %>%
    mutate(Time_to_deg = ifelse(is.infinite(Time_to_deg), NA, Time_to_deg))
  
  cat("\nTime to AMP < 10 ug/mL by replicate:\n")
  print(deg_time_abs_sp)
  
  # Wilcoxon only if both groups have at least one non-NA value
  x_pre <- deg_time_abs_sp$Time_to_deg[deg_time_abs_sp$Evo == "Pre-exposed"]
  y_anc <- deg_time_abs_sp$Time_to_deg[deg_time_abs_sp$Evo == "Ancestral"]
  
  x_pre <- x_pre[!is.na(x_pre)]
  y_anc <- y_anc[!is.na(y_anc)]
  
  cat("\nWilcoxon test for earlier degradation in pre-exposed:\n")
  if (length(x_pre) > 0 & length(y_anc) > 0) {
    print(wilcox.test(
      x = x_pre,
      y = y_anc,
      alternative = "less",
      exact = TRUE,
      correct = FALSE
    ))
  } else {
    cat("Not run: one or both groups never reached <10 ug/mL.\n")
  }
  
  # degradation status at 6 h
  deg6_abs_sp <- dat_sp %>%
    filter(Timeh == 6) %>%
    mutate(Degraded10 = AMP_conc < 10)
  
  cat("\nCounts at 6 h (<10 ug/mL):\n")
  tab6_sp <- table(deg6_abs_sp$Evo, deg6_abs_sp$Degraded10)
  print(tab6_sp)
  
  cat("\nFisher's exact test at 6 h (pre-exposed more often <10 ug/mL):\n")
  if (all(dim(tab6_sp) == c(2, 2))) {
    print(fisher.test(tab6_sp, alternative = "greater"))
  } else {
    cat("Not run: contingency table is not 2 x 2.\n")
  }
  
  # OD comparison at 6 h
  cat("\nWelch t-test for OD at 6 h:\n")
  if (nrow(deg6_abs_sp) > 0 && length(unique(deg6_abs_sp$Evo)) == 2) {
    print(t.test(OD ~ Evo, data = deg6_abs_sp, var.equal = FALSE))
  } else {
    cat("Not run: insufficient 6 h data.\n")
  }
  
  # correlation between time to degradation and OD at 6 h
  summ_merge_sp <- left_join(
    deg_time_abs_sp,
    deg6_abs_sp %>% select(Evo, Replicate, OD6 = OD),
    by = c("Evo", "Replicate")
  )
  
  cat("\nSpearman correlation: degradation time vs OD at 6 h:\n")
  ok <- complete.cases(summ_merge_sp[, c("Time_to_deg", "OD6")])
  if (sum(ok) >= 3) {
    print(cor.test(
      summ_merge_sp$Time_to_deg[ok],
      summ_merge_sp$OD6[ok],
      method = "spearman"
    ))
  } else {
    cat("Not run: too few complete observations.\n")
  }
}


Strain: HAMBI_403 

Time to AMP < 10 ug/mL by replicate:
# A tibble: 6 × 3
  Evo         Replicate Time_to_deg
  <fct>           <int>       <dbl>
1 Ancestral           1           6
2 Ancestral           2           6
3 Ancestral           3           6
4 Pre-exposed         1           6
5 Pre-exposed         2           6
6 Pre-exposed         3           6

Wilcoxon test for earlier degradation in pre-exposed:


Warning message in wilcox.test.default(x = x_pre, y = y_anc, alternative = "less", :
“cannot compute exact p-value with ties”



	Wilcoxon rank sum test

data:  x_pre and y_anc
W = 4.5, p-value = NA
alternative hypothesis: true location shift is less than 0


Counts at 6 h (<10 ug/mL):
             
              TRUE
  Ancestral      3
  Pre-exposed    3

Fisher's exact test at 6 h (pre-exposed more often <10 ug/mL):
Not run: contingency table is not 2 x 2.

Welch t-test for OD at 6 h:

	Welch Two Sample t-test

data:  OD by Evo
t = 1.6742, df = 3.0966, p-value = 0.1898
alternative hypothesis: true difference in means between group Ancestral and group Pre-exposed is not equal to 0
95 percent confidence interval:
 -0.01388462  0.04588462
sample estimates:
  mean in group Ancestral mean in group Pre-exposed 
                    0.073                     0.057 


Spearman correlation: degradation time vs OD at 6 h:


Warning message in cor(rank(x), rank(y)):
“the standard deviation is zero”



	Spearman's rank correlation rho

data:  summ_merge_sp$Time_to_deg[ok] and summ_merge_sp$OD6[ok]
S = NA, p-value = NA
alternative hypothesis: true rho is not equal to 0
sample estimates:
rho 
 NA 


Strain: HAMBI_1977 

Time to AMP < 10 ug/mL by replicate:
# A tibble: 6 × 3
  Evo         Replicate Time_to_deg
  <fct>           <int>       <dbl>
1 Ancestral           1          24
2 Ancestral           2          24
3 Ancestral           3          24
4 Pre-exposed         1          24
5 Pre-exposed         2          24
6 Pre-exposed         3          24

Wilcoxon test for earlier degradation in pre-exposed:


Warning message in wilcox.test.default(x = x_pre, y = y_anc, alternative = "less", :
“cannot compute exact p-value with ties”



	Wilcoxon rank sum test

data:  x_pre and y_anc
W = 4.5, p-value = NA
alternative hypothesis: true location shift is less than 0


Counts at 6 h (<10 ug/mL):
             
              FALSE
  Ancestral       3
  Pre-exposed     3

Fisher's exact test at 6 h (pre-exposed more often <10 ug/mL):
Not run: contingency table is not 2 x 2.

Welch t-test for OD at 6 h:

	Welch Two Sample t-test

data:  OD by Evo
t = 4.726, df = 2.053, p-value = 0.03989
alternative hypothesis: true difference in means between group Ancestral and group Pre-exposed is not equal to 0
95 percent confidence interval:
 0.005549377 0.093783956
sample estimates:
  mean in group Ancestral mean in group Pre-exposed 
               0.10700000                0.05733333 


Spearman correlation: degradation time vs OD at 6 h:


Warning message in cor(rank(x), rank(y)):
“the standard deviation is zero”



	Spearman's rank correlation rho

data:  summ_merge_sp$Time_to_deg[ok] and summ_merge_sp$OD6[ok]
S = NA, p-value = NA
alternative hypothesis: true rho is not equal to 0
sample estimates:
rho 
 NA 



In [8]:
# =========================================================
# Compact summary
# =========================================================

compact_summary <- function(sp) {
  dat_sp <- read.table("../data/AMP_key_species.txt", header=TRUE, sep="\t", dec=",") %>%
    filter(Species == sp) %>%
    mutate(
      Evo   = factor(Evo, levels = c("ANC", "EVO"),
                     labels = c("Ancestral", "Pre-exposed")),
      Timeh = as.numeric(Timeh),
      AMP_conc = as.numeric(AMP_conc),
      OD = as.numeric(OD)
    )
  
  deg_time <- dat_sp %>%
    group_by(Evo, Replicate) %>%
    summarise(Time_to_deg = min(Timeh[AMP_conc < 10], na.rm = TRUE), .groups = "drop") %>%
    mutate(Time_to_deg = ifelse(is.infinite(Time_to_deg), NA, Time_to_deg))
  
  deg6 <- dat_sp %>%
    filter(Timeh == 6) %>%
    mutate(Degraded10 = AMP_conc < 10)
  
  anc_med <- median(deg_time$Time_to_deg[deg_time$Evo == "Ancestral"], na.rm = TRUE)
  pre_med <- median(deg_time$Time_to_deg[deg_time$Evo == "Pre-exposed"], na.rm = TRUE)
  
  anc_deg6 <- sum(deg6$Degraded10[deg6$Evo == "Ancestral"], na.rm = TRUE)
  anc_n6   <- sum(deg6$Evo == "Ancestral")
  pre_deg6 <- sum(deg6$Degraded10[deg6$Evo == "Pre-exposed"], na.rm = TRUE)
  pre_n6   <- sum(deg6$Evo == "Pre-exposed")
  
  cat("\n", sp, ":\n", sep = "")
  cat("median time to <10 ug/mL  | ancestral =", anc_med,
      ", pre-exposed =", pre_med, "\n")
  cat("6 h degraded (<10 ug/mL)  | ancestral =", anc_deg6, "/", anc_n6,
      ", pre-exposed =", pre_deg6, "/", pre_n6, "\n")
}

compact_summary("HAMBI_403")
compact_summary("HAMBI_1977")


HAMBI_403:
median time to <10 ug/mL  | ancestral = 6 , pre-exposed = 6 
6 h degraded (<10 ug/mL)  | ancestral = 3 / 3 , pre-exposed = 3 / 3 

HAMBI_1977:
median time to <10 ug/mL  | ancestral = 24 , pre-exposed = 24 
6 h degraded (<10 ug/mL)  | ancestral = 0 / 3 , pre-exposed = 0 / 3 


In [9]:
# =========================================================
# MIC-k correlations and simple count summary
# =========================================================

# species means by population
trait_species <- df3 %>%
  group_by(strainID, Pop) %>%
  summarise(
    log10MIC_mean = mean(log10MIC, na.rm = TRUE),
    k_mean        = mean(k, na.rm = TRUE),
    .groups = "drop"
  )

trait_anc <- trait_species %>% filter(Pop == "ANC")
trait_evo <- trait_species %>% filter(Pop == "EVO")

# 1) MIC-k correlation within ancestral species
cat("\n============================================\n")
cat("MIC-k correlation across species: ancestral\n")
cat("============================================\n")
cor_anc <- cor.test(trait_anc$log10MIC_mean, trait_anc$k_mean, method = "spearman", exact = FALSE)
print(cor_anc)

# 2) MIC-k correlation within pre-exposed species
cat("\n============================================\n")
cat("MIC-k correlation across species: pre-exposed\n")
cat("============================================\n")
cor_evo <- cor.test(trait_evo$log10MIC_mean, trait_evo$k_mean, method = "spearman", exact = FALSE)
print(cor_evo)

# 3) correlation between evolutionary changes in MIC and k
trait_change <- trait_species %>%
  pivot_wider(names_from = Pop, values_from = c(log10MIC_mean, k_mean)) %>%
  mutate(
    dMIC = log10MIC_mean_EVO - log10MIC_mean_ANC,
    dk   = k_mean_EVO - k_mean_ANC
  )

cat("\n============================================\n")
cat("Correlation between change in MIC and change in k\n")
cat("============================================\n")
cor_change <- cor.test(trait_change$dMIC, trait_change$dk, method = "spearman", exact = FALSE)
print(cor_change)

# 4) simple counts for manuscript text
trait_change <- trait_change %>%
  mutate(
    MIC_increased = dMIC > 0,
    k_decreased   = dk < 0,
    cost_without_resistance_gain = (dk < 0 & dMIC <= 0)
  )

cat("\n============================================\n")
cat("Simple species counts\n")
cat("============================================\n")
cat("Species with increased MIC:", sum(trait_change$MIC_increased, na.rm = TRUE), "/", nrow(trait_change), "\n")
cat("Species with decreased k:", sum(trait_change$k_decreased, na.rm = TRUE), "/", nrow(trait_change), "\n")
cat("Species with cost without resistance gain:", sum(trait_change$cost_without_resistance_gain, na.rm = TRUE), "/", nrow(trait_change), "\n")

cat("\nSpecies with cost without resistance gain:\n")
print(trait_change %>%
  filter(cost_without_resistance_gain) %>%
  select(strainID, dMIC, dk) %>%
  arrange(dk))

# optional: compact table for supplement
cat("\n============================================\n")
cat("Per-species evolutionary change table\n")
cat("============================================\n")
print(trait_change %>%
  select(strainID, log10MIC_mean_ANC, log10MIC_mean_EVO, dMIC,
         k_mean_ANC, k_mean_EVO, dk) %>%
  arrange(desc(dMIC)))

# species with both increased MIC and decreased k
both_tradeoff <- sum(trait_change$MIC_increased & trait_change$k_decreased, na.rm = TRUE)

cat("Species with increased MIC and decreased k:", both_tradeoff, "/", nrow(trait_change), "\n")

cat("\nSpecies with both increased MIC and decreased k:\n")
print(trait_change %>%
  filter(MIC_increased & k_decreased) %>%
  select(strainID, dMIC, dk) %>%
  arrange(dk))

# =========================================================
# Partition species responses
# =========================================================

both_tradeoff <- sum(trait_change$MIC_increased & trait_change$k_decreased, na.rm = TRUE)
res_only      <- sum(trait_change$MIC_increased & !trait_change$k_decreased, na.rm = TRUE)
cost_only     <- sum(!trait_change$MIC_increased & trait_change$k_decreased, na.rm = TRUE)

cat("\n============================================\n")
cat("Partition of species responses\n")
cat("============================================\n")
cat("Increased MIC + decreased k (trade-off):", both_tradeoff, "/", nrow(trait_change), "\n")
cat("Increased MIC without cost:", res_only, "/", nrow(trait_change), "\n")
cat("Cost without increased MIC:", cost_only, "/", nrow(trait_change), "\n")

# optional: list species in each category
cat("\nSpecies with increased MIC without cost:\n")
print(trait_change %>%
  filter(MIC_increased & !k_decreased) %>%
  select(strainID, dMIC, dk))

cat("\nSpecies with cost without increased MIC:\n")
print(trait_change %>%
  filter(!MIC_increased & k_decreased) %>%
  select(strainID, dMIC, dk))


MIC-k correlation across species: ancestral

	Spearman's rank correlation rho

data:  trait_anc$log10MIC_mean and trait_anc$k_mean
S = 1734, p-value = 0.5143
alternative hypothesis: true rho is not equal to 0
sample estimates:
      rho 
0.1432806 


MIC-k correlation across species: pre-exposed

	Spearman's rank correlation rho

data:  trait_evo$log10MIC_mean and trait_evo$k_mean
S = 1730, p-value = 0.5084
alternative hypothesis: true rho is not equal to 0
sample estimates:
      rho 
0.1452569 


Correlation between change in MIC and change in k

	Spearman's rank correlation rho

data:  trait_change$dMIC and trait_change$dk
S = 1080, p-value = 0.02487
alternative hypothesis: true rho is not equal to 0
sample estimates:
      rho 
0.4664032 


Simple species counts
Species with increased MIC: 19 / 23 
Species with decreased k: 15 / 23 
Species with cost without resistance gain: 3 / 23 

Species with cost without resistance gain:
# A tibble: 3 × 3
  strainID     dMIC      dk
  <chr>  

In [10]:
### Proportion of species with increase in MIC and decrease in k

table(
  MIC_increase = trait_change$dMIC > 0,
  k_decrease   = trait_change$dk < 0
)

prop_tradeoff <- mean(
  trait_change$dMIC > 0 &
  trait_change$dk < 0
)

prop_tradeoff

            k_decrease
MIC_increase FALSE TRUE
       FALSE     1    3
       TRUE      7   12

[1] 0.5217391

In [11]:
### Supplementary table on MIC and k and their change

supp_table <- df3 %>%
  group_by(strainID, Pop) %>%
  summarise(
    MIC_mean = mean(MIC, na.rm = TRUE),
    k_mean = mean(k, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  tidyr::pivot_wider(
    names_from = Pop,
    values_from = c(MIC_mean, k_mean)
  ) %>%
  mutate(
    dMIC = MIC_mean_EVO - MIC_mean_ANC,
    dk   = k_mean_EVO - k_mean_ANC
  ) %>%
  arrange(MIC_mean_ANC)

print(supp_table)

write.table(
  supp_table,
  "../data/Supplementary_Table_MIC_k_changes.txt",
  sep = "\t",
  row.names = FALSE,
  quote = FALSE
)

# A tibble: 23 × 7
   strainID   MIC_mean_ANC MIC_mean_EVO k_mean_ANC k_mean_EVO     dMIC      dk
   <chr>             <dbl>        <dbl>      <dbl>      <dbl>    <dbl>   <dbl>
 1 HAMBI-1988        0            0.301      0         0.332    0.301   0.332 
 2 HAMBI-2443        0.267        0.334      0.659     0.626    0.0668 -0.0326
 3 HAMBI-2792        0.267        0.902      0.927     0.634    0.635  -0.293 
 4 HAMBI-3237        0.267        0          0.473     0       -0.267  -0.473 
 5 HAMBI-2160        0.401        7.42       0.155     0.0582   7.02   -0.0970
 6 HAMBI-3031        0.585        2.03       0.310     0.187    1.45   -0.123 
 7 HAMBI-1279        0.601      117.         0.164     0.218  116.      0.0545
 8 HAMBI-1292        0.601        0.401      0.291     0.279   -0.200  -0.0111
 9 HAMBI-0097        1.13         0.601      1.12      0.542   -0.526  -0.576 
10 HAMBI-1842        1.35         1.35       0.415     0.700    0       0.285 
# ℹ 13 more rows


In [12]:
############################################################
# 1. Bounded-IC50 pipeline
############################################################

library(dplyr)
library(stringr)
library(minpack.lm)

df_ic50 <- read.table(
  "../data/AMP_growth_k_auc.txt",
  header = TRUE,
  check.names = FALSE,
  sep = "\t"
) %>%
  filter(Species != "only medium") %>%
  mutate(
    strainID = paste0(
      "HAMBI-",
      str_pad(
        as.character(Species),
        4,
        pad = "0"
      )
    ),
    AB = as.numeric(AB),
    k = as.numeric(k),

    # Same operational filtering used earlier in the notebook:
    # values below 0.1 or above 1.3 are treated as no growth.
    k = if_else(
      !is.finite(k) |
        k < 0.1 |
        k > 1.3,
      0,
      k
    )
  ) %>%
  select(
    strainID,
    Pop,
    Replicate,
    AB,
    k
  ) %>%
  filter(
    is.finite(AB),
    AB >= 0
  )


############################################################
# 2. MIC helper
############################################################

estimate_curve_MIC <- function(dat) {

  d <- dat %>%
    group_by(AB) %>%
    summarise(
      k = median(
        k,
        na.rm = TRUE
      ),
      .groups = "drop"
    ) %>%
    arrange(AB)

  if (nrow(d) == 0L) {
    return(NA_real_)
  }

  # Identify the first occurrence of two successive
  # no-growth measurements.
  first_zero_pair <- NA_integer_

  if (nrow(d) >= 2L) {

    pair_starts <- which(
      d$k[-nrow(d)] <= 0 &
        d$k[-1L] <= 0
    )

    if (length(pair_starts) > 0L) {
      first_zero_pair <- pair_starts[1L]
    }
  }

  # Ignore isolated positive measurements after the first
  # confirmed no-growth region.
  growth_region <- if (is.finite(first_zero_pair)) {

    if (first_zero_pair > 1L) {

      d[
        seq_len(first_zero_pair - 1L),
        ,
        drop = FALSE
      ]

    } else {

      d[
        0,
        ,
        drop = FALSE
      ]
    }

  } else {

    d
  }

  positive_AB <- growth_region$AB[
    growth_region$k > 0
  ]

  # No positive growth at any usable concentration.
  if (length(positive_AB) == 0L) {

    zero_AB <- d$AB[
      d$k <= 0
    ]

    return(
      if (length(zero_AB) > 0L) {
        min(zero_AB)
      } else {
        min(d$AB)
      }
    )
  }

  highest_growth <- max(positive_AB)

  higher_tested <- d$AB[
    d$AB > highest_growth
  ]

  # MIC is the next measured concentration above the
  # highest concentration with growth. If growth occurs at
  # the maximum tested concentration, use that maximum.
  if (length(higher_tested) > 0L) {
    min(higher_tested)
  } else {
    max(d$AB)
  }
}


############################################################
# 3. estimate_bounded_ic50 function
############################################################

estimate_bounded_ic50 <- function(
    dat,
    target = 0.5,
    reversal_tolerance = 0.15,
    minimum_doses_for_Hill = 4L) {

  # Collapse any technical duplicates at the same dose.
  d <- dat %>%
    mutate(
      AB = as.numeric(AB),
      k = as.numeric(k)
    ) %>%
    filter(
      is.finite(AB),
      is.finite(k),
      AB >= 0
    ) %>%
    group_by(AB) %>%
    summarise(
      k = median(
        k,
        na.rm = TRUE
      ),
      .groups = "drop"
    ) %>%
    arrange(AB)

  n_doses <- nrow(d)

  # Standardised output used by all exit paths.
  make_result <- function(
      IC50_rep = NA_real_,
      MIC_rep = NA_real_,
      method = "no_positive_growth",
      baseline_source = NA_character_,
      k0 = NA_real_,
      n_reversals = 0L,
      RMSE = NA_real_) {

    tibble(
      IC50_rep = as.numeric(IC50_rep),
      MIC_rep = as.numeric(MIC_rep),
      method = as.character(method),
      baseline_source = as.character(
        baseline_source
      ),
      k0 = as.numeric(k0),
      n_doses = as.integer(n_doses),
      n_reversals = as.integer(
        n_reversals
      ),
      RMSE = as.numeric(RMSE)
    )
  }

  if (n_doses == 0L) {
    return(make_result())
  }

  MIC_rep <- estimate_curve_MIC(d)

  if (!is.finite(MIC_rep)) {
    MIC_rep <- max(d$AB)
  }


  ##########################################################
  # Establish the no-antibiotic baseline
  ##########################################################

  if (any(d$AB == 0)) {

    k0 <- median(
      d$k[d$AB == 0],
      na.rm = TRUE
    )

    baseline_source <- "AB_zero"

  } else {

    lowest_AB <- min(d$AB)

    k0 <- median(
      d$k[d$AB == lowest_AB],
      na.rm = TRUE
    )

    baseline_source <- "lowest_dose"
  }


  ##########################################################
  # Handle replicates without usable positive growth
  ##########################################################

  # A finite zero is retained because the population-level
  # summary in Part #6 explicitly handles operational zeros.
  if (
    !is.finite(k0) ||
      k0 <= 0 ||
      !any(d$k > 0)
  ) {

    return(
      make_result(
        IC50_rep = 0,
        MIC_rep = MIC_rep,
        method = "no_positive_growth",
        baseline_source = baseline_source,
        k0 = k0
      )
    )
  }


  ##########################################################
  # Relative growth and diagnostic reversals
  ##########################################################

  d <- d %>%
    mutate(
      rel_k = pmax(
        k / k0,
        0
      )
    )

  # Fit only through the estimated MIC. This prevents
  # isolated positive values after confirmed no growth from
  # influencing the IC50.
  d_fit <- d %>%
    filter(
      AB <= MIC_rep +
        sqrt(.Machine$double.eps)
    )

  if (nrow(d_fit) < 2L) {
    d_fit <- d
  }

  # Count substantial increases in relative growth as dose
  # increases. These are retained as diagnostics.
  n_reversals <- sum(
    diff(d_fit$rel_k) >
      reversal_tolerance,
    na.rm = TRUE
  )


  ##########################################################
  # Monotonic isotonic estimate
  ##########################################################

  # isoreg() estimates an increasing relationship.
  # Fitting -rel_k therefore produces a decreasing curve
  # after changing the sign back.
  isotonic_fit <- stats::isoreg(
    d_fit$AB,
    -d_fit$rel_k
  )

  isotonic_y <- -isotonic_fit$yf

  crossing <- which(
    isotonic_y <= target
  )

  if (length(crossing) == 0L) {

    # The response has not fallen below 50% before the MIC.
    # Use the MIC as the upper bounded estimate.
    isotonic_IC50 <- MIC_rep

  } else {

    crossing_index <- crossing[1L]

    if (crossing_index == 1L) {

      isotonic_IC50 <- d_fit$AB[1L]

    } else {

      x1 <- d_fit$AB[
        crossing_index - 1L
      ]

      x2 <- d_fit$AB[
        crossing_index
      ]

      y1 <- isotonic_y[
        crossing_index - 1L
      ]

      y2 <- isotonic_y[
        crossing_index
      ]

      # Linear interpolation around relative growth = 0.5.
      isotonic_IC50 <- if (
        abs(y2 - y1) <
          sqrt(.Machine$double.eps)
      ) {

        x2

      } else {

        x1 +
          (target - y1) *
          (x2 - x1) /
          (y2 - y1)
      }
    }
  }

  isotonic_IC50 <- min(
    max(isotonic_IC50, 0),
    MIC_rep
  )

  isotonic_RMSE <- sqrt(
    mean(
      (
        d_fit$rel_k -
          isotonic_y
      )^2
    )
  )


  ##########################################################
  # Bounded Hill model
  ##########################################################

  positive_doses <- d_fit$AB[
    d_fit$AB > 0
  ]

  can_fit_Hill <-
    nrow(d_fit) >=
      minimum_doses_for_Hill &&
    length(unique(positive_doses)) >= 2L &&
    is.finite(MIC_rep) &&
    MIC_rep > 0

  if (can_fit_Hill) {

    lower_IC50 <- max(
      min(positive_doses) / 1000,
      sqrt(.Machine$double.eps)
    )

    upper_IC50 <- max(
      MIC_rep,
      lower_IC50 * 1.01
    )

    start_IC50 <- isotonic_IC50

    if (
      !is.finite(start_IC50) ||
        start_IC50 <= lower_IC50 ||
        start_IC50 >= upper_IC50
    ) {

      start_IC50 <- sqrt(
        lower_IC50 *
          upper_IC50
      )
    }

    Hill_fit <- tryCatch(
      suppressWarnings(
        minpack.lm::nlsLM(
          rel_k ~
            1 /
            (
              1 +
                (AB / IC50)^slope
            ),
          data = d_fit,
          start = list(
            IC50 = start_IC50,
            slope = 1
          ),
          lower = c(
            IC50 = lower_IC50,
            slope = 0.05
          ),
          upper = c(
            IC50 = upper_IC50,
            slope = 50
          ),
          control =
            minpack.lm::nls.lm.control(
              maxiter = 500
            )
        )
      ),
      error = function(e) NULL
    )

    fit_ok <-
      !is.null(Hill_fit) &&
      all(
        is.finite(
          coef(Hill_fit)
        )
      ) &&
      (
        is.null(
          Hill_fit$convInfo$isConv
        ) ||
          isTRUE(
            Hill_fit$convInfo$isConv
          )
      )

    if (fit_ok) {

      IC50_rep <- unname(
        coef(Hill_fit)[["IC50"]]
      )

      predicted <- predict(
        Hill_fit,
        newdata = d_fit
      )

      RMSE <- sqrt(
        mean(
          (
            d_fit$rel_k -
              predicted
          )^2
        )
      )

      # Hard safeguard required by Part #5.
      IC50_rep <- min(
        max(IC50_rep, 0),
        MIC_rep
      )

      return(
        make_result(
          IC50_rep = IC50_rep,
          MIC_rep = MIC_rep,
          method = "bounded_Hill",
          baseline_source =
            baseline_source,
          k0 = k0,
          n_reversals =
            n_reversals,
          RMSE = RMSE
        )
      )
    }
  }


  ##########################################################
  # Hill failure or insufficient doses: isotonic fallback
  ##########################################################

  make_result(
    IC50_rep = isotonic_IC50,
    MIC_rep = MIC_rep,
    method = "isotonic_fallback",
    baseline_source = baseline_source,
    k0 = k0,
    n_reversals = n_reversals,
    RMSE = isotonic_RMSE
  )
}

In [13]:
############################################################
# 4. Replicate-level IC50 estimates
############################################################

IC50_replicate <- df_ic50 %>%
  group_by(
    strainID,
    Pop,
    Replicate
  ) %>%
  group_modify(
    ~ estimate_bounded_ic50(.x)
  ) %>%
  ungroup()


print(IC50_replicate)


############################################################
# 5. Check that no replicate IC50 exceeds MIC
############################################################

IC50_above_MIC <- IC50_replicate %>%
  filter(
    is.finite(IC50_rep),
    is.finite(MIC_rep),
    IC50_rep > MIC_rep + 1e-8
  )

stopifnot(
  nrow(IC50_above_MIC) == 0
)


############################################################
# 6. One IC50 value per species and population
############################################################

IC50_species <- IC50_replicate %>%
  group_by(
    strainID,
    Pop
  ) %>%
  summarise(
    n_replicates = n(),

    n_Hill = sum(
      method == "bounded_Hill",
      na.rm = TRUE
    ),

    n_isotonic = sum(
      method == "isotonic_fallback",
      na.rm = TRUE
    ),

    n_no_growth = sum(
      method == "no_positive_growth",
      na.rm = TRUE
    ),

    # Use every finite replicate estimate, including when
    # only one replicate yielded a usable estimate.
    IC50 = {
      values <- IC50_rep[
        is.finite(IC50_rep)
      ]

      if (length(values) == 0) {

        NA_real_

      } else if (all(values > 0)) {

        # Geometric mean for positive concentration values
        exp(
          mean(
            log(values)
          )
        )

      } else {

        # Handles operational zero values
        median(values)
      }
    },

    # Match the MIC aggregation used for the species/population.
    MIC_mean = mean(
      MIC_rep,
      na.rm = TRUE
    ),

    median_RMSE = median(
      RMSE,
      na.rm = TRUE
    ),

    .groups = "drop"
  ) %>%
  mutate(
    # Final species-level safeguard
    IC50 = if_else(
      is.finite(MIC_mean),
      pmin(IC50, MIC_mean),
      IC50
    )
  )


print(IC50_species)


############################################################
# 7. Compact ANC/EVO table
############################################################

IC50_table <- IC50_species %>%
  select(
    strainID,
    Pop,
    IC50
  ) %>%
  pivot_wider(
    names_from = Pop,
    values_from = IC50,
    names_prefix = "IC50_"
  )

print(IC50_table)


############################################################
# 8. Optional three-significant-digit display
############################################################

IC50_table_display <- IC50_table %>%
  mutate(
    across(
      starts_with("IC50_"),
      ~ ifelse(
        is.finite(.x),
        formatC(
          .x,
          digits = 3,
          format = "g"
        ),
        "NA"
      )
    )
  )

print(IC50_table_display)

# A tibble: 92 × 11
   strainID   Pop   Replicate IC50_rep  MIC_rep method     baseline_source    k0
   <chr>      <chr>     <int>    <dbl>    <dbl> <chr>      <chr>           <dbl>
 1 HAMBI-0006 ANC           1    4.37    52.0   bounded_H… AB_zero         0.860
 2 HAMBI-0006 ANC           2    9.03    34.7   bounded_H… AB_zero         0.783
 3 HAMBI-0006 EVO           1 2000     2000     isotonic_… AB_zero         0.365
 4 HAMBI-0006 EVO           2  369.    2000     bounded_H… AB_zero         0.786
 5 HAMBI-0097 ANC           1    0.391    0.902 bounded_H… AB_zero         1.03 
 6 HAMBI-0097 ANC           2    0.489    1.35  bounded_H… AB_zero         1.21 
 7 HAMBI-0097 EVO           1    0.429    0.601 bounded_H… AB_zero         0.556
 8 HAMBI-0097 EVO           2    0.418    0.601 bounded_H… AB_zero         0.527
 9 HAMBI-0105 ANC           1    4.97     6.85  bounded_H… AB_zero         0.610
10 HAMBI-0105 ANC           2    5.01     6.85  bounded_H… AB_zero         0.585
# ℹ 82 m

In [14]:
IC50_species

strainID,Pop,n_replicates,n_Hill,n_isotonic,n_no_growth,IC50,MIC_mean,median_RMSE
<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>
HAMBI-0006,ANC,2,2,0,0,6.282477e+00,43.3538248,2.274549e-01
HAMBI-0006,EVO,2,1,1,0,8.589827e+02,2000.0000000,2.679689e-01
HAMBI-0097,ANC,2,2,0,0,4.372259e-01,1.1277325,1.613419e-01
HAMBI-0097,EVO,2,2,0,0,4.233219e-01,0.6014573,5.534747e-02
HAMBI-0105,ANC,2,2,0,0,4.990727e+00,6.8509748,1.101696e-01
HAMBI-0105,EVO,2,2,0,0,8.390696e+00,8.5637185,3.417876e-01
HAMBI-0262,ANC,2,2,0,0,2.029918e+00,2.0299185,3.682606e-01
HAMBI-0262,EVO,2,2,0,0,8.390696e+00,8.5637185,2.851461e-01
HAMBI-0403,ANC,2,2,0,0,5.061749e+02,592.5925926,1.725826e-01


In [15]:
############################################################
# EMPIRICALLY ANCHORED IC50 USING THE EXACT SI MIC PROTOCOL
#
# Key rules
# ---------
# 1. MIC is calculated PER REPLICATE using exactly the SI rule:
#      k < 0.1 or k > 1.3 -> 0 for MIC classification;
#      ignore positive values after the first two successive zeros;
#      MIC = next measured AB above the last retained growth dose.
# 2. MIC is calculated BEFORE IC50 point cleaning and is never
#    re-estimated from pooled or smoothed data.
# 3. IC50 uses the unthresholded valid k values (except invalid >1.3),
#    a shared robust baseline checked across AB=0, the lowest positive
#    dose, and both replicates.
# 4. Isolated low/zero or high gradient points are discarded only when
#    contradicted by the other replicate and/or neighbouring doses.
# 5. IC50 is estimated separately for each replicate from the empirical
#    monotone half-growth crossing, then positive replicate IC50 values
#    are combined with a geometric mean.
# 6. Strong unresolved replicate disagreement gives NA, not an arbitrary
#    selected estimate. Right-censored curves are reported as >= max dose.
############################################################

library(dplyr)
library(tidyr)
library(purrr)
library(stringr)

############################################################
# 1. Read data and create separate MIC and IC50 responses
############################################################

df_raw <- read.table(
  "../data/AMP_growth_k_auc.txt",
  header = TRUE,
  check.names = FALSE,
  sep = "\t"
)

df_ic50 <- df_raw %>%
  filter(Species != "only medium") %>%
  mutate(
    strainID = paste0(
      "HAMBI-",
      str_pad(as.character(Species), 4, pad = "0")
    ),
    Replicate = as.character(Replicate),
    AB = as.numeric(AB),
    k_raw = as.numeric(k),

    # EXACT response used by the SI MIC calculation.
    k_MIC = if_else(
      !is.finite(k_raw) | k_raw < 0.1 | k_raw > 1.3,
      0,
      k_raw
    ),

    # Response used for IC50. Values below 0.1 are retained because
    # they contain information about the approach to half growth.
    # Values above 1.3 are assay-invalid and are not treated as zeros.
    k_fit = case_when(
      !is.finite(k_raw) ~ NA_real_,
      k_raw > 1.3       ~ NA_real_,
      k_raw < 0         ~ 0,
      TRUE              ~ k_raw
    )
  ) %>%
  select(strainID, Pop, Replicate, AB, k_raw, k_MIC, k_fit) %>%
  filter(is.finite(AB), AB >= 0)


############################################################
# 2. Exact replicate-level SI MIC
############################################################

estimate_SI_MIC <- function(dat) {
  a <- dat %>%
    group_by(AB) %>%
    summarise(k_MIC = median(k_MIC, na.rm = TRUE), .groups = "drop") %>%
    arrange(AB)

  full_AB <- sort(unique(a$AB))

  if (length(full_AB) == 0L) {
    return(tibble(
      MIC_rep = NA_real_,
      MIC_censored_rep = NA,
      first_confirmed_zero = NA_real_,
      last_growth_AB = NA_real_
    ))
  }

  first_confirmed_zero <- NA_real_

  if (nrow(a) >= 2L) {
    pair_start <- which(
      head(a$k_MIC, -1L) == 0 &
        tail(a$k_MIC, -1L) == 0
    )[1L]

    if (!is.na(pair_start)) {
      first_confirmed_zero <- a$AB[pair_start]
    }
  }

  a_growth_region <- if (is.finite(first_confirmed_zero)) {
    a %>% filter(AB < first_confirmed_zero)
  } else {
    a
  }

  growth_AB <- a_growth_region$AB[a_growth_region$k_MIC > 0]

  if (length(growth_AB) == 0L) {
    MIC_rep <- if (is.finite(first_confirmed_zero)) {
      first_confirmed_zero
    } else {
      min(full_AB)
    }
    last_growth_AB <- NA_real_
  } else {
    last_growth_AB <- max(growth_AB)
    next_AB <- full_AB[full_AB > last_growth_AB]

    MIC_rep <- if (length(next_AB) > 0L) {
      min(next_AB)
    } else {
      max(full_AB)
    }
  }

  # Right-censored when growth remains at the largest tested dose and
  # no confirmed terminal zero region exists.
  maximum_AB <- max(full_AB)
  growth_at_maximum <- any(a$AB == maximum_AB & a$k_MIC > 0)
  MIC_censored_rep <-
    !is.finite(first_confirmed_zero) &&
    isTRUE(growth_at_maximum) &&
    isTRUE(all.equal(MIC_rep, maximum_AB))

  tibble(
    MIC_rep = MIC_rep,
    MIC_censored_rep = MIC_censored_rep,
    first_confirmed_zero = first_confirmed_zero,
    last_growth_AB = last_growth_AB
  )
}

MIC_replicate <- df_ic50 %>%
  group_by(strainID, Pop, Replicate) %>%
  group_modify(~ estimate_SI_MIC(.x)) %>%
  ungroup()

# This reproduces the SI population-level MIC aggregation.
MIC_species <- MIC_replicate %>%
  group_by(strainID, Pop) %>%
  summarise(
    MIC_mean = mean(MIC_rep, na.rm = TRUE),
    n_MIC_censored = sum(MIC_censored_rep %in% TRUE, na.rm = TRUE),
    .groups = "drop"
  )

# Exact population means expected from the existing SI MIC table/code.
# The script stops immediately if the replicate-level MIC implementation
# does not reproduce these values (within numerical tolerance).
expected_SI_MIC <- tribble(
  ~strainID,    ~Pop, ~expected_MIC,
  "HAMBI-0006", "ANC",   43.3538248,
  "HAMBI-0006", "EVO", 2000.0000000,
  "HAMBI-0097", "ANC",    1.1277325,
  "HAMBI-0097", "EVO",    0.6014573,
  "HAMBI-0105", "ANC",    6.8509748,
  "HAMBI-0105", "EVO",    8.5637185,
  "HAMBI-0262", "ANC",    2.0299185,
  "HAMBI-0262", "EVO",    8.5637185,
  "HAMBI-0403", "ANC",  592.5925926,
  "HAMBI-0403", "EVO",  493.8271605,
  "HAMBI-1279", "ANC",    0.6014573,
  "HAMBI-1279", "EVO",  117.0553269,
  "HAMBI-1287", "ANC",   19.2683666,
  "HAMBI-1287", "EVO",  175.5829904,
  "HAMBI-1292", "ANC",    0.6014573,
  "HAMBI-1292", "EVO",    0.4009715,
  "HAMBI-1299", "ANC",    3.2986175,
  "HAMBI-1299", "EVO",  175.5829904,
  "HAMBI-1842", "ANC",    1.3532790,
  "HAMBI-1842", "EVO",    1.3532790,
  "HAMBI-1896", "ANC",   52.0245897,
  "HAMBI-1896", "EVO", 1333.3333330,
  "HAMBI-1972", "ANC",  219.4787380,
  "HAMBI-1972", "EVO", 2000.0000000,
  "HAMBI-1977", "ANC",  962.9629628,
  "HAMBI-1977", "EVO", 2000.0000000,
  "HAMBI-1988", "ANC",    0.0000000,
  "HAMBI-1988", "EVO",    0.3007287,
  "HAMBI-2159", "ANC", 2000.0000000,
  "HAMBI-2159", "EVO", 2000.0000000,
  "HAMBI-2160", "ANC",    0.4009715,
  "HAMBI-2160", "EVO",    7.4218893,
  "HAMBI-2164", "ANC",    4.5673165,
  "HAMBI-2164", "EVO",   52.0245897,
  "HAMBI-2443", "ANC",    0.2673144,
  "HAMBI-2443", "EVO",    0.3341430,
  "HAMBI-2494", "ANC",    4.5673165,
  "HAMBI-2494", "EVO",  117.0553269,
  "HAMBI-2659", "ANC",    7.4218893,
  "HAMBI-2659", "EVO",    7.4218893,
  "HAMBI-2792", "ANC",    0.2673144,
  "HAMBI-2792", "EVO",    0.9021860,
  "HAMBI-3031", "ANC",    0.5847502,
  "HAMBI-3031", "EVO",    2.0299185,
  "HAMBI-3237", "ANC",    0.2673144,
  "HAMBI-3237", "EVO",    0.0000000
)

MIC_validation <- MIC_species %>%
  left_join(expected_SI_MIC, by = c("strainID", "Pop")) %>%
  mutate(absolute_difference = abs(MIC_mean - expected_MIC))

if (any(MIC_validation$absolute_difference > 1e-5, na.rm = TRUE)) {
  print(MIC_validation %>% filter(absolute_difference > 1e-5), n = Inf)
  stop("MIC implementation does not reproduce the SI MIC values.")
}


############################################################
# 3. Helpers
############################################################

geometric_interpolation <- function(x1, y1, x2, y2, target = 0.5) {
  if (!all(is.finite(c(x1, y1, x2, y2)))) return(NA_real_)
  if (abs(y2 - y1) < sqrt(.Machine$double.eps)) return(x2)

  f <- (target - y1) / (y2 - y1)
  f <- min(max(f, 0), 1)

  if (x1 > 0 && x2 > 0) {
    exp(log(x1) + f * (log(x2) - log(x1)))
  } else {
    x1 + f * (x2 - x1)
  }
}

weighted_pava_increasing <- function(y, w = rep(1, length(y))) {
  stopifnot(length(y) == length(w))
  if (length(y) == 0L) return(numeric())

  blocks <- lapply(seq_along(y), function(i) {
    list(first = i, last = i, weight = w[i], mean = y[i])
  })

  repeat {
    if (length(blocks) <= 1L) break
    means <- vapply(blocks, `[[`, numeric(1), "mean")
    violation <- which(head(means, -1L) > tail(means, -1L))[1L]
    if (is.na(violation)) break

    left <- blocks[[violation]]
    right <- blocks[[violation + 1L]]
    total_weight <- left$weight + right$weight

    merged <- list(
      first = left$first,
      last = right$last,
      weight = total_weight,
      mean = (left$mean * left$weight + right$mean * right$weight) /
        total_weight
    )

    blocks <- append(
      blocks[-c(violation, violation + 1L)],
      list(merged),
      after = violation - 1L
    )
  }

  fitted <- numeric(length(y))
  for (block in blocks) {
    fitted[block$first:block$last] <- block$mean
  }
  fitted
}

weighted_pava_decreasing <- function(y, w = rep(1, length(y))) {
  -weighted_pava_increasing(-y, w)
}

count_reversals <- function(ab, rel, tolerance = 0.25) {
  keep <- is.finite(ab) & is.finite(rel)
  if (sum(keep) < 2L) return(0L)
  ord <- order(ab[keep])
  as.integer(sum(diff(rel[keep][ord]) > tolerance, na.rm = TRUE))
}

choose_high_consensus_cluster <- function(values, fold_tolerance = 1.75) {
  values <- values[is.finite(values) & values > 0]
  n <- length(values)
  if (n == 0L) return(NA_real_)

  for (size in seq.int(n, 1L, by = -1L)) {
    subsets <- combn(seq_len(n), size, simplify = FALSE)
    valid <- keep(subsets, function(index) {
      x <- values[index]
      max(x) / min(x) <= fold_tolerance
    })

    if (length(valid) > 0L) {
      medians <- map_dbl(valid, ~ median(values[.x]))
      return(median(values[valid[[which.max(medians)]]]))
    }
  }

  NA_real_
}


############################################################
# 4. Shared robust baseline and baseline-failure detection
############################################################

estimate_shared_baseline <- function(dat) {
  positive_AB <- sort(unique(dat$AB[dat$AB > 0]))
  lowest_AB <- if (length(positive_AB)) positive_AB[1L] else NA_real_

  early <- dat %>%
    filter(AB %in% c(0, lowest_AB), is.finite(k_fit)) %>%
    group_by(Replicate, AB) %>%
    summarise(k_fit = median(k_fit), .groups = "drop")

  replicate_qc <- dat %>%
    distinct(Replicate) %>%
    left_join(
      early %>%
        group_by(Replicate) %>%
        summarise(
          control_k = {
            x <- k_fit[AB == 0]
            if (length(x)) median(x) else NA_real_
          },
          lowest_dose_k = {
            x <- k_fit[AB == lowest_AB]
            if (length(x)) median(x) else NA_real_
          },
          early_proxy = {
            x <- k_fit[is.finite(k_fit)]
            if (length(x)) max(x) else NA_real_
          },
          .groups = "drop"
        ),
      by = "Replicate"
    )

  finite_proxy <- replicate_qc$early_proxy[
    is.finite(replicate_qc$early_proxy) & replicate_qc$early_proxy > 0
  ]

  if (!length(finite_proxy)) {
    return(list(
      baseline = NA_real_,
      valid = FALSE,
      lowest_AB = lowest_AB,
      replicate_qc = replicate_qc %>% mutate(baseline_failure = TRUE),
      reason = "no positive early-growth measurements"
    ))
  }

  healthiest <- max(finite_proxy)

  replicate_qc <- replicate_qc %>%
    mutate(
      baseline_failure =
        !is.finite(early_proxy) |
        early_proxy < healthiest / 2.5,
      baseline_reason = case_when(
        !is.finite(early_proxy) ~ "missing AB=0 and lowest-dose growth",
        baseline_failure ~ "early growth >2.5-fold below healthier replicate",
        TRUE ~ NA_character_
      )
    )

  retained <- replicate_qc$Replicate[!replicate_qc$baseline_failure]
  early_values <- early$k_fit[early$Replicate %in% retained]
  baseline <- choose_high_consensus_cluster(early_values, 1.75)

  list(
    baseline = baseline,
    valid = is.finite(baseline) && baseline >= 0.1,
    lowest_AB = lowest_AB,
    replicate_qc = replicate_qc,
    reason = if (is.finite(baseline) && baseline >= 0.1) {
      "robust shared baseline"
    } else {
      "robust baseline below 0.1"
    }
  )
}


############################################################
# 5. Conservative removal of isolated gradient anomalies
############################################################

clean_gradient_points <- function(points) {
  points <- points %>%
    mutate(
      keep = is.finite(k_fit) & !baseline_failure,
      discard_reason = if_else(
        baseline_failure,
        "replicate baseline failure",
        NA_character_
      )
    ) %>%
    arrange(Replicate, AB)

  # Points at or after the SI-defined MIC are not used as quantitative
  # IC50 observations. A zero anchor is added later at MIC when uncensored.
  points <- points %>%
    mutate(
      beyond_MIC =
        !MIC_censored_rep &
        is.finite(MIC_rep) &
        AB >= MIC_rep,
      keep = keep & !beyond_MIC,
      discard_reason = case_when(
        beyond_MIC ~ "at or beyond replicate SI MIC",
        TRUE ~ discard_reason
      )
    )

  for (iteration in seq_len(5L)) {
    removed_this_pass <- FALSE

    for (i in which(points$keep)) {
      focal <- points[i, ]
      support <- numeric()

      # Other replicate at the same dose.
      support <- c(
        support,
        points$rel[
          points$keep &
            points$AB == focal$AB &
            points$Replicate != focal$Replicate
        ]
      )

      # Nearest lower and higher retained measurements in the same replicate.
      same_rep <- points %>%
        filter(keep, Replicate == focal$Replicate, AB != focal$AB) %>%
        arrange(AB)

      lower <- same_rep %>% filter(AB < focal$AB)
      higher <- same_rep %>% filter(AB > focal$AB)
      if (nrow(lower)) support <- c(support, tail(lower$rel, 1L))
      if (nrow(higher)) support <- c(support, head(higher$rel, 1L))

      support <- support[is.finite(support)]
      if (length(support) < 2L) next

      expected <- median(support)
      n_high <- sum(support >= 0.45)
      n_low <- sum(support <= 0.15)

      isolated_low <-
        focal$rel <= 0.15 && expected >= 0.45 && n_high >= 2L

      isolated_high <-
        focal$rel >= 0.45 && expected <= 0.15 && n_low >= 2L

      general_outlier <-
        length(support) >= 3L &&
        abs(focal$rel - expected) >= 0.40 &&
        sum(abs(support - expected) <= 0.20) >= 2L

      if (isolated_low || isolated_high || general_outlier) {
        points$keep[i] <- FALSE
        points$discard_reason[i] <- if (isolated_low) {
          "isolated low/zero contradicted by replicate and neighbours"
        } else if (isolated_high) {
          "isolated high contradicted by replicate and neighbours"
        } else {
          "isolated local gradient discrepancy"
        }
        removed_this_pass <- TRUE
      }
    }

    if (!removed_this_pass) break
  }

  points
}


############################################################
# 6. Replicate-level empirical IC50
############################################################

estimate_replicate_IC50 <- function(rep_points, baseline, mic_row) {
  replicate_id <- unique(rep_points$Replicate)
  MIC_rep <- mic_row$MIC_rep[1L]
  MIC_censored_rep <- mic_row$MIC_censored_rep[1L]

  usable <- rep_points %>%
    filter(keep, AB > 0) %>%
    group_by(AB) %>%
    summarise(
      k_observed = median(k_fit),
      rel_observed = median(rel),
      .groups = "drop"
    ) %>%
    arrange(AB)

  curve <- bind_rows(
    tibble(
      AB = 0,
      k_observed = baseline,
      rel_observed = 1,
      weight = 4,
      point_type = "shared robust baseline"
    ),
    usable %>%
      mutate(weight = 1, point_type = "retained observation")
  )

  if (is.finite(MIC_rep) && !isTRUE(MIC_censored_rep)) {
    curve <- curve %>% filter(AB < MIC_rep)
    curve <- bind_rows(
      curve,
      tibble(
        AB = MIC_rep,
        k_observed = 0,
        rel_observed = 0,
        weight = 4,
        point_type = "SI MIC zero anchor"
      )
    )
  }

  curve <- curve %>%
    arrange(AB) %>%
    group_by(AB) %>%
    summarise(
      k_observed = weighted.mean(k_observed, weight),
      rel_observed = weighted.mean(rel_observed, weight),
      weight = sum(weight),
      point_type = paste(unique(point_type), collapse = "; "),
      .groups = "drop"
    )

  if (nrow(curve) < 2L) {
    return(tibble(
      Replicate = replicate_id,
      MIC_rep = MIC_rep,
      MIC_censored_rep = MIC_censored_rep,
      IC50_rep = NA_real_,
      IC50_censored_rep = NA,
      half_bracket_low_AB = NA_real_,
      half_bracket_high_AB = NA_real_,
      half_bracket_low_k = NA_real_,
      half_bracket_high_k = NA_real_,
      empirical_RMSE = NA_real_,
      low_resolution = NA,
      method = "not_estimable",
      replicate_status = "fewer than two usable curve points"
    ))
  }

  curve$rel_fit <- weighted_pava_decreasing(
    pmin(pmax(curve$rel_observed, 0), 1.5),
    curve$weight
  )
  curve$rel_fit[curve$AB == 0] <- 1
  if (!isTRUE(MIC_censored_rep)) {
    curve$rel_fit[curve$AB == MIC_rep] <- 0
  }
  curve$k_fit_monotone <- curve$rel_fit * baseline

  observed <- grepl("retained observation", curve$point_type)
  RMSE <- if (any(observed)) {
    sqrt(weighted.mean(
      (curve$rel_observed[observed] - curve$rel_fit[observed])^2,
      curve$weight[observed]
    ))
  } else {
    NA_real_
  }

  crossing <- which(curve$rel_fit <= 0.5)[1L]

  if (is.na(crossing)) {
    return(tibble(
      Replicate = replicate_id,
      MIC_rep = MIC_rep,
      MIC_censored_rep = MIC_censored_rep,
      IC50_rep = NA_real_,
      IC50_censored_rep = TRUE,
      half_bracket_low_AB = max(curve$AB),
      half_bracket_high_AB = NA_real_,
      half_bracket_low_k = tail(curve$k_fit_monotone, 1L),
      half_bracket_high_k = NA_real_,
      empirical_RMSE = RMSE,
      low_resolution = FALSE,
      method = "right_censored",
      replicate_status = "growth never fell to half the robust baseline"
    ))
  }

  if (crossing == 1L) {
    return(tibble(
      Replicate = replicate_id,
      MIC_rep = MIC_rep,
      MIC_censored_rep = MIC_censored_rep,
      IC50_rep = NA_real_,
      IC50_censored_rep = NA,
      half_bracket_low_AB = NA_real_,
      half_bracket_high_AB = curve$AB[1L],
      half_bracket_low_k = NA_real_,
      half_bracket_high_k = curve$k_fit_monotone[1L],
      empirical_RMSE = RMSE,
      low_resolution = NA,
      method = "not_estimable",
      replicate_status = "baseline itself at or below half-growth threshold"
    ))
  }

  lo <- crossing - 1L
  hi <- crossing

  IC50_rep <- geometric_interpolation(
    curve$AB[lo], curve$rel_fit[lo],
    curve$AB[hi], curve$rel_fit[hi],
    target = 0.5
  )

  tibble(
    Replicate = replicate_id,
    MIC_rep = MIC_rep,
    MIC_censored_rep = MIC_censored_rep,
    IC50_rep = IC50_rep,
    IC50_censored_rep = FALSE,
    half_bracket_low_AB = curve$AB[lo],
    half_bracket_high_AB = curve$AB[hi],
    half_bracket_low_k = curve$k_fit_monotone[lo],
    half_bracket_high_k = curve$k_fit_monotone[hi],
    empirical_RMSE = RMSE,
    low_resolution = curve$AB[lo] == 0,
    method = "empirical_monotone_interpolation",
    replicate_status = if (curve$AB[lo] == 0) {
      "half-growth bracket begins at AB=0; low concentration resolution"
    } else {
      "estimated inside observed half-growth bracket"
    }
  )
}


############################################################
# 7. Analyse one strain-population group
############################################################

analyse_strain_population <- function(dat, mic_group) {
  baseline_result <- estimate_shared_baseline(dat)
  baseline <- baseline_result$baseline

  if (!baseline_result$valid) {
    return(list(
      replicate = tibble(),
      points = tibble(),
      summary = tibble(
        baseline = baseline,
        IC50 = NA_real_,
        IC50_label = "NA",
        method = "not_estimable",
        QC_status = paste("baseline QC failed:", baseline_result$reason)
      )
    ))
  }

  points <- dat %>%
    left_join(
      baseline_result$replicate_qc %>%
        select(Replicate, baseline_failure, baseline_reason),
      by = "Replicate"
    ) %>%
    left_join(mic_group, by = "Replicate") %>%
    mutate(rel = k_fit / baseline) %>%
    filter(AB > 0)

  cleaned <- clean_gradient_points(points)

  replicate_gradient_qc <- cleaned %>%
    group_by(Replicate) %>%
    summarise(
      n_preMIC_points = sum(!beyond_MIC),
      n_isolated_discarded = sum(
        !keep & !beyond_MIC & !baseline_failure,
        na.rm = TRUE
      ),
      discarded_fraction = if_else(
        n_preMIC_points > 0,
        n_isolated_discarded / n_preMIC_points,
        1
      ),
      n_reversals = count_reversals(AB[keep], rel[keep]),
      .groups = "drop"
    ) %>%
    mutate(
      gradient_failure =
        (n_isolated_discarded >= 2L & discarded_fraction >= 0.30) |
        n_reversals >= 3L,
      gradient_reason = case_when(
        n_reversals >= 3L ~ "three or more large monotonic reversals",
        n_isolated_discarded >= 2L & discarded_fraction >= 0.30 ~
          "too many isolated gradient discrepancies",
        TRUE ~ NA_character_
      ),
      quality_score = n_isolated_discarded + n_reversals +
        4 * as.integer(gradient_failure)
    )

  cleaned <- cleaned %>%
    left_join(
      replicate_gradient_qc %>%
        select(Replicate, gradient_failure, gradient_reason, quality_score),
      by = "Replicate"
    ) %>%
    mutate(
      keep = keep & !gradient_failure,
      discard_reason = case_when(
        gradient_failure ~ gradient_reason,
        TRUE ~ discard_reason
      )
    )

  rep_results <- map_dfr(unique(dat$Replicate), function(rep_id) {
    base_fail <- baseline_result$replicate_qc$baseline_failure[
      baseline_result$replicate_qc$Replicate == rep_id
    ]
    grad_fail <- replicate_gradient_qc$gradient_failure[
      replicate_gradient_qc$Replicate == rep_id
    ]

    mic_row <- mic_group %>% filter(Replicate == rep_id)

    if (isTRUE(base_fail) || isTRUE(grad_fail)) {
      return(tibble(
        Replicate = rep_id,
        MIC_rep = mic_row$MIC_rep[1L],
        MIC_censored_rep = mic_row$MIC_censored_rep[1L],
        IC50_rep = NA_real_,
        IC50_censored_rep = NA,
        half_bracket_low_AB = NA_real_,
        half_bracket_high_AB = NA_real_,
        half_bracket_low_k = NA_real_,
        half_bracket_high_k = NA_real_,
        empirical_RMSE = NA_real_,
        low_resolution = NA,
        method = "discarded_replicate",
        replicate_status = if (isTRUE(base_fail)) {
          "discarded: baseline growth failure"
        } else {
          paste("discarded:", replicate_gradient_qc$gradient_reason[
            replicate_gradient_qc$Replicate == rep_id
          ])
        }
      ))
    }

    estimate_replicate_IC50(
      cleaned %>% filter(Replicate == rep_id),
      baseline,
      mic_row
    )
  }) %>%
    left_join(
      replicate_gradient_qc %>% select(Replicate, quality_score),
      by = "Replicate"
    ) %>%
    mutate(quality_score = replace_na(quality_score, 99))

  # Resolve strong replicate-level IC50 disagreement conservatively.
  finite_rows <- which(
    is.finite(rep_results$IC50_rep) &
      rep_results$IC50_rep > 0 &
      rep_results$method != "discarded_replicate"
  )
  censored_rows <- which(rep_results$IC50_censored_rep %in% TRUE)

  conflict <- FALSE
  discarded_for_conflict <- integer()

  if (length(finite_rows) == 2L) {
    ratio <- max(rep_results$IC50_rep[finite_rows]) /
      min(rep_results$IC50_rep[finite_rows])

    if (ratio > 4) {
      q <- rep_results$quality_score[finite_rows]
      if (abs(diff(q)) >= 2) {
        discarded_for_conflict <- finite_rows[which.max(q)]
      } else {
        conflict <- TRUE
      }
    }
  }

  if (length(finite_rows) == 1L && length(censored_rows) == 1L) {
    finite_i <- finite_rows[1L]
    cens_i <- censored_rows[1L]
    maximum_tested <- max(dat$AB, na.rm = TRUE)

    if (rep_results$IC50_rep[finite_i] < maximum_tested / 4) {
      if (
        rep_results$quality_score[finite_i] >=
          rep_results$quality_score[cens_i] + 2
      ) {
        discarded_for_conflict <- finite_i
      } else if (
        rep_results$quality_score[cens_i] >=
          rep_results$quality_score[finite_i] + 2
      ) {
        discarded_for_conflict <- cens_i
      } else {
        conflict <- TRUE
      }
    }
  }

  if (length(discarded_for_conflict)) {
    rep_results$method[discarded_for_conflict] <- "discarded_replicate"
    rep_results$replicate_status[discarded_for_conflict] <-
      "discarded: strong IC50 discrepancy and poorer independent QC"
    rep_results$IC50_rep[discarded_for_conflict] <- NA_real_
    rep_results$IC50_censored_rep[discarded_for_conflict] <- NA
  }

  retained_finite <- rep_results %>%
    filter(is.finite(IC50_rep), IC50_rep > 0, method != "discarded_replicate")
  retained_censored <- rep_results %>%
    filter(IC50_censored_rep %in% TRUE, method != "discarded_replicate")

  maximum_tested <- max(dat$AB, na.rm = TRUE)

  if (conflict) {
    IC50 <- NA_real_
    label <- "NA"
    method <- "not_estimable"
    status <- "unresolved strong disagreement between otherwise similar-quality replicates"
  } else if (nrow(retained_finite) > 0L && nrow(retained_censored) == 0L) {
    IC50 <- exp(mean(log(retained_finite$IC50_rep)))
    label <- format(signif(IC50, 3), trim = TRUE, scientific = FALSE)
    method <- "geometric_mean_of_empirical_replicate_IC50"
    status <- paste0(
      nrow(retained_finite), " finite replicate IC50 estimate(s) retained"
    )
  } else if (nrow(retained_finite) == 0L && nrow(retained_censored) > 0L) {
    IC50 <- NA_real_
    label <- paste0("≥", format(signif(maximum_tested, 3), trim = TRUE))
    method <- "right_censored"
    status <- "all retained replicates remained above half growth at the maximum dose"
  } else if (nrow(retained_finite) > 0L && nrow(retained_censored) > 0L) {
    IC50 <- NA_real_
    label <- "NA"
    method <- "not_estimable"
    status <- "mixed finite and right-censored replicate IC50 results"
  } else {
    IC50 <- NA_real_
    label <- "NA"
    method <- "not_estimable"
    status <- "no retained replicate-level IC50 estimate"
  }

  list(
    replicate = rep_results,
    points = cleaned,
    summary = tibble(
      baseline = baseline,
      IC50 = IC50,
      IC50_label = label,
      method = method,
      QC_status = status
    )
  )
}


############################################################
# 8. Run all strain-population groups
############################################################

MIC_nested <- MIC_replicate %>%
  group_by(strainID, Pop) %>%
  nest(
    MIC_data = c(
      Replicate, MIC_rep, MIC_censored_rep,
      first_confirmed_zero, last_growth_AB
    )
  )

analysis_nested <- df_ic50 %>%
  group_by(strainID, Pop) %>%
  nest() %>%
  left_join(MIC_nested, by = c("strainID", "Pop")) %>%
  mutate(result = map2(data, MIC_data, analyse_strain_population))

IC50_replicate <- analysis_nested %>%
  transmute(
    strainID,
    Pop,
    replicate = map(result, "replicate")
  ) %>%
  unnest(replicate)

IC50_cleaned_points <- analysis_nested %>%
  transmute(
    strainID,
    Pop,
    points = map(result, "points")
  ) %>%
  unnest(points)

IC50_species <- analysis_nested %>%
  transmute(
    strainID,
    Pop,
    summary = map(result, "summary")
  ) %>%
  unnest(summary) %>%
  left_join(MIC_species, by = c("strainID", "Pop")) %>%
  left_join(
    IC50_replicate %>%
      group_by(strainID, Pop) %>%
      summarise(
        n_replicates = n(),
        n_replicates_used = sum(method != "discarded_replicate"),
        n_replicates_discarded = sum(method == "discarded_replicate"),
        n_right_censored = sum(IC50_censored_rep %in% TRUE, na.rm = TRUE),
        median_RMSE = median(empirical_RMSE, na.rm = TRUE),
        .groups = "drop"
      ) %>%
      mutate(median_RMSE = if_else(is.nan(median_RMSE), NA_real_, median_RMSE)),
    by = c("strainID", "Pop")
  ) %>%
  select(
    strainID, Pop,
    n_replicates, n_replicates_used, n_replicates_discarded,
    n_right_censored,
    IC50, IC50_label,
    MIC_mean, n_MIC_censored,
    baseline, median_RMSE,
    method, QC_status
  )


############################################################
# 9. Supplementary-table order and paste-ready output
############################################################

supplementary_order <- c(
  "HAMBI-1988", "HAMBI-2443", "HAMBI-2792", "HAMBI-3237",
  "HAMBI-2160", "HAMBI-3031", "HAMBI-1279", "HAMBI-1292",
  "HAMBI-0097", "HAMBI-1842", "HAMBI-0262", "HAMBI-1299",
  "HAMBI-2164", "HAMBI-2494", "HAMBI-0105", "HAMBI-2659",
  "HAMBI-1287", "HAMBI-0006", "HAMBI-1896", "HAMBI-1972",
  "HAMBI-0403", "HAMBI-1977", "HAMBI-2159"
)

IC50_supplementary <- IC50_species %>%
  mutate(
    strainID = factor(strainID, levels = supplementary_order),
    Pop = factor(Pop, levels = c("ANC", "EVO"))
  ) %>%
  arrange(strainID, Pop) %>%
  select(
    strainID, Pop,
    IC50, IC50_label,
    MIC_mean,
    n_replicates_used,
    n_replicates_discarded,
    n_right_censored,
    median_RMSE,
    QC_status
  )

print(MIC_replicate, n = Inf)
print(IC50_replicate, n = Inf, width = Inf)
print(IC50_supplementary, n = Inf, width = Inf)

# Optional exports:
# write.table(MIC_replicate, "../data/MIC_replicate_SI_protocol.txt",
#             sep = "\t", quote = FALSE, row.names = FALSE)
# write.table(IC50_replicate, "../data/IC50_replicate_empirical.txt",
#             sep = "\t", quote = FALSE, row.names = FALSE)
# write.table(IC50_supplementary, "../data/IC50_supplementary_paste_ready.txt",
#             sep = "\t", quote = FALSE, row.names = FALSE)

# A tibble: 92 × 7
   strainID   Pop   Replicate  MIC_rep MIC_censored_rep first_confirmed_zero
   <chr>      <chr> <chr>        <dbl> <lgl>                           <dbl>
 1 HAMBI-0006 ANC   1           52.0   FALSE                          52.0  
 2 HAMBI-0006 ANC   2           34.7   FALSE                          34.7  
 3 HAMBI-0006 EVO   1         2000     TRUE                           NA    
 4 HAMBI-0006 EVO   2         2000     FALSE                          NA    
 5 HAMBI-0097 ANC   1            0.902 FALSE                           0.902
 6 HAMBI-0097 ANC   2            1.35  FALSE                           1.35 
 7 HAMBI-0097 EVO   1            0.601 FALSE                           0.601
 8 HAMBI-0097 EVO   2            0.601 FALSE                           0.601
 9 HAMBI-0105 ANC   1            6.85  FALSE                           6.85 
10 HAMBI-0105 ANC   2            6.85  FALSE                           6.85 
11 HAMBI-0105 EVO   1            6.85  FALSE             

In [16]:
head(IC50_supplementary)

strainID,Pop,IC50,IC50_label,MIC_mean,n_replicates_used,n_replicates_discarded,n_right_censored,median_RMSE,QC_status
<fct>,<fct>,<dbl>,<chr>,<dbl>,<int>,<int>,<int>,<dbl>,<chr>
HAMBI-1988,ANC,NA,NA,0.0000000,NA,NA,NA,NA,baseline QC failed: robust baseline below 0.1
HAMBI-1988,EVO,0.4225819,0.423,0.3007287,1,1,0,0.0000000,1 finite replicate IC50 estimate(s) retained
HAMBI-2443,ANC,0.1336572,0.134,0.2673144,2,0,0,NA,2 finite replicate IC50 estimate(s) retained
HAMBI-2443,EVO,0.1465465,0.147,0.3341430,2,0,0,0.0000000,2 finite replicate IC50 estimate(s) retained
HAMBI-2792,ANC,0.1336572,0.134,0.2673144,2,0,0,NA,2 finite replicate IC50 estimate(s) retained
HAMBI-2792,EVO,0.6842799,0.684,0.9021860,2,0,0,0.1168136,2 finite replicate IC50 estimate(s) retained


In [17]:
# Fisher's exact test based on the classifications
# reported in Supplementary Table S1 (including some rounding of low values).
#
# Twenty-one species had paired MIC and carrying-capacity estimates.
# Of these:
#   - 15 showed increased MIC
#   - 14 showed decreased carrying capacity (k)
#   - 10 showed both changes
#
# Therefore:
#   - MIC increased and k decreased     = 10
#   - MIC increased and k not decreased = 15 - 10 = 5
#   - MIC not increased and k decreased = 14 - 10 = 4
#   - Neither change                     = 21 - 10 - 5 - 4 = 2

sign_tab <- matrix(
  c(
    2, 4,
    5, 10
  ),
  nrow = 2,
  byrow = TRUE,
  dimnames = list(
    MIC_increased = c("no", "yes"),
    k_decreased   = c("no", "yes")
  )
)

cat("\nObserved combinations:\n")
print(sign_tab)

cat("\nSummary counts:\n")
cat("Species included:", sum(sign_tab), "\n")
cat("MIC increased:", sum(sign_tab["yes", ]), "\n")
cat("k decreased:", sum(sign_tab[, "yes"]), "\n")
cat("MIC increased and k decreased:", sign_tab["yes", "yes"], "\n")

cat("\nFisher's exact test:\n")
print(fisher.test(sign_tab))


Observed combinations:
             k_decreased
MIC_increased no yes
          no   2   4
          yes  5  10

Summary counts:
Species included: 21 
MIC increased: 15 
k decreased: 14 
MIC increased and k decreased: 10 

Fisher's exact test:

	Fisher's Exact Test for Count Data

data:  sign_tab
p-value = 1
alternative hypothesis: true odds ratio is not equal to 1
95 percent confidence interval:
  0.06807161 10.32399749
sample estimates:
odds ratio 
         1 

